In [1]:
from decoder_unidirectional import VCDecoder, run_training
from vc_dataset import VCDataset
import torchinfo
import yaml

CONFIG_PATH = "configs.yaml"


def load_config(config_path=CONFIG_PATH):
    with open(config_path, "r", encoding="utf-8") as config_file:
        return yaml.safe_load(config_file)

decoder = VCDecoder()
vc_dataset = VCDataset(config=load_config())

if vc_dataset is not None:
    print("VCDataset loaded successfully.")
    print("VCDataset Info:")
    print(f"Number of samples in the dataset: {len(vc_dataset)}")

print("=" * 67)
if decoder is not None:
    print("Unidirectional decoder loaded successfully.")
    print("Decoder Info:")
    print(torchinfo.summary(decoder))

VCDataset loaded successfully.
VCDataset Info:
Number of samples in the dataset: 44057
Unidirectional decoder loaded successfully.
Decoder Info:
Layer (type:depth-idx)                   Param #
VCDecoder                                --
├─LSTM: 1-1                              3,018,752
├─ModuleList: 1-2                        --
│    └─Sequential: 2-1                   --
│    │    └─ConvNorm: 3-1                1,311,232
│    │    └─BatchNorm1d: 3-2             1,024
│    │    └─ReLU: 3-3                    --
│    │    └─Dropout: 3-4                 --
│    └─Sequential: 2-2                   --
│    │    └─ConvNorm: 3-5                1,311,232
│    │    └─BatchNorm1d: 3-6             1,024
│    │    └─ReLU: 3-7                    --
│    │    └─Dropout: 3-8                 --
│    └─Sequential: 2-3                   --
│    │    └─ConvNorm: 3-9                1,311,232
│    │    └─BatchNorm1d: 3-10            1,024
│    │    └─ReLU: 3-11                   --
│    │    └─Dropout: 

In [2]:
# Establish training parameters (in configs[training])

if config := load_config():
    print("Configuration loaded successfully.")
    print("Configuration Info:")
    for key, value in config.items():
        if key == "training":
            print(f"{key}:")
            for sub_key, sub_value in value.items():
                print(f"  {sub_key}: {sub_value}")

Configuration loaded successfully.
Configuration Info:
training:
  smoke_test_training: True
  batch_size: 16
  num_epochs: 5
  learning_rate: 0.001
  weight_decay: 0.0001
  save_interval: 1
  model_checkpoint_dir: model_checkpoints/


In [5]:
# Training
config = load_config()

# Notebook-side overrides for this first real run.
# This does not modify configs.yaml on disk.
config["training"]["smoke_test_training"] = False
config["training"]["num_epochs"] = 5
CHECKPOINT_PATH = "model_checkpoints/decoder_unidirectional_epoch_0002.pt"

train_dataset = VCDataset(config=config, split="train")
val_dataset = VCDataset(config=config, split="val")
test_dataset = VCDataset(config=config, split="test")

print("Unidirectional training run setup")
print("=" * 67)
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(f"Test samples:  {len(test_dataset)}")
print()
print("Training parameters")
print("=" * 67)
for key, value in config["training"].items():
    print(f"{key}: {value}")

model, optimizer, training_info = run_training(
    config=config,
    dataset=train_dataset,
    val_dataset=val_dataset,
    model=decoder,
    checkpoint_path=CHECKPOINT_PATH,
)

training_history = training_info["history"]
training_history

Unidirectional training run setup
Train samples: 35164
Val samples:   4501
Test samples:  4392

Training parameters
smoke_test_training: False
batch_size: 16
num_epochs: 5
learning_rate: 0.001
weight_decay: 0.0001
save_interval: 1
model_checkpoint_dir: model_checkpoints/
Loaded checkpoint: model_checkpoints\decoder_unidirectional_epoch_0002.pt (epoch 2, next epoch 3, optimizer loaded: True)


Training:   0%|          | 0/2198 [00:00<?, ?it/s]

Validation:   0%|          | 0/282 [00:00<?, ?it/s]

Epoch 3/5 - train loss: 0.5978 - val loss: 0.6484
Saved checkpoint: model_checkpoints\decoder_unidirectional_epoch_0003.pt


Training:   0%|          | 0/2198 [00:00<?, ?it/s]

KeyboardInterrupt: 